# Motor Vehicle Collisions — time range

Reads the zipped CSV and prints the earliest and latest crash timestamps.

In [1]:
import pandas as pd

# pandas reads a single-file zip directly; only load the date/time columns to keep it fast
df = pd.read_csv(
    "data/motor_vehicle_collisions.zip",
    usecols=["CRASH DATE", "CRASH TIME"],
)
print(f"{len(df):,} rows")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df["CRASH DATETIME"] = pd.to_datetime(
    df["CRASH DATE"] + " " + df["CRASH TIME"],
    format="%m/%d/%Y %H:%M",
)

start, end = df["CRASH DATETIME"].min(), df["CRASH DATETIME"].max()
print(f"Start: {start}")
print(f"End:   {end}")
print(f"Span:  {end - start}")

## Distribution of crash dates

Crashes per month across the whole range. If dates were roughly uniform, the line would sit flat around the dashed mean.

In [ ]:
import matplotlib.pyplot as plt

monthly = df.set_index("CRASH DATETIME").resample("MS").size()

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(monthly.index, monthly.values, color="#2a78d6", alpha=0.15, linewidth=0)
ax.plot(monthly.index, monthly.values, color="#2a78d6", linewidth=2)
ax.axhline(monthly.mean(), color="#52514e", linewidth=1, linestyle="--")
ax.annotate(f"mean {monthly.mean():,.0f}", (monthly.index[-1], monthly.mean()),
            xytext=(4, 4), textcoords="offset points", color="#52514e", fontsize=9, ha="right")

ax.set_title("Crashes per month", loc="left")
ax.set_ylabel("Crashes")
ax.set_ylim(bottom=0)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.grid(axis="y", color="#e5e4e0", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.show()

In [ ]:
# Quick numeric check: how much do monthly counts vary?
print(monthly.describe().round(0).to_string())
print(f"\nCoefficient of variation: {monthly.std() / monthly.mean():.1%}")
print(f"Busiest month:  {monthly.idxmax():%Y-%m} ({monthly.max():,})")
print(f"Quietest month: {monthly.idxmin():%Y-%m} ({monthly.min():,})")